# 🧠 CT/MRI Registration Pipeline — LS-DYNA `.k` Mesh Version
> **SimpleITK only | No ANTs | Supports `.k` LS-DYNA Mesh Input/Output**

| Cell | Stage | Description | Est. Time |
|------|-------|-------------|-----------|
| 1 | Install | Dependencies | once/session |
| 2 | Config | Set file paths | < 5 sec |
| 3 | K-Reader | Parse `.k` mesh → voxel volume | < 30 sec |
| 4 | Helpers | Load functions | < 5 sec |
| 5 | Preprocess | Resample → Normalize → Center align | < 30 sec |
| 6 | Rigid | 6-DOF registration | ~1 min |
| 7 | Affine | 12-DOF refinement | ~1 min |
| 8 | Deformable | Demons non-linear warp | ~1 min |
| 9 | Apply to Mesh | Warp `.k` node coordinates → new `.k` | < 30 sec |
| 10 | Validation | Visual + Dice score | < 30 sec |

**Total: ~4–5 minutes** ✅  
**Run in order top → bottom. Never skip a cell.**

### How `.k` files are handled
- **Input CT** : `.nii.gz` voxel volume (fixed image — unchanged)
- **Input MRI / Moving** : LS-DYNA `.k` mesh file  
  → Node coordinates are extracted, voxelized into a binary volume for registration  
  → After registration the final transform is applied back to every node  
  → A new `*_registered.k` file is written with updated node XYZ positions

## ⚙️ Cell 1 — Install Dependencies
Run once per session.

In [2]:
pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.1/532.1 MB 15.9 MB/s  0:00:33 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 15.4 MB/s  0:00:21 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 18.9 MB/s  0:00:09 eta 0:00:010:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 18.0 MB/s  0:00:11 eta 0:00:010:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 17.6 MB/s  0:00:03 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.6/197.6 MB 15.6 MB/s  0:00:12 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 18.1 MB/s  0:00:000.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 16.5 MB/s  0:00:26 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 15.9 MB/s  0:00:006.4 MB/s eta 0:00:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 15.1 MB/s  0:00:05 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!pip install SimpleITK nibabel matplotlib numpy vtk -q

import torch
print("✅ GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU")

ModuleNotFoundError: No module named 'torch'

## 📁 Cell 2 — User Config
Edit the paths below. `MRI_PATH` now points to a `.k` mesh file.

In [3]:
import pyvista as pv

mesh = pv.read(r"C:\Mai\BME\BioDat Work\2D-3D-Mri-image\New_version\Data\ImageToStl.com_head_voxels.stl")
mesh.save("head_model.vtk")        # or .vtp for the XML version

In [4]:
# After parsing nodes/elements with our parse_k_file()
import pyvista as pv
import numpy as np

# nodes_registered = your warped node array (N x 3)
cloud = pv.PolyData(nodes_registered)
cloud.save("registered_surface.vtk")

NameError: name 'nodes_registered' is not defined

In [ ]:
import os

# ── USER EDIT: set your file paths here ──────────────────────────────────────
CT_PATH    = r'C:\Mai\BME\BioDat Work\2D-3D-Mri-image\New_version\Data\outputs\vtk_export\ImageToStl.com_head_voxels.vtu'  # fixed image  (CT voxel volume)
MRI_PATH   = r'C:\Mai\BME\BioDat Work\2D-3D-Mri-image\New_version\Code\outputs\vtk_export\Head_V6.vtu'                    # moving image (LS-DYNA .k mesh)
OUTPUT_DIR = r'outputs/case01'                                       # all outputs saved here
# ─────────────────────────────────────────────────────────────────────────────

# Voxelization resolution (mm) — smaller = finer volume, slower
VOXEL_SIZE_MM = 1.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output folder : {OUTPUT_DIR}")
print(f"   CT path       : {CT_PATH}   exists={os.path.exists(CT_PATH)}")
print(f"   .k mesh path  : {MRI_PATH}  exists={os.path.exists(MRI_PATH)}")

✅ Output folder : outputs/case01
   CT path       : /Users/lapastang/Documents/head-Model-label.nii.gz   exists=False
   .k mesh path  : D:/2d_3d_regis dataset/Head_V6.k  exists=True


## 🔩 Cell 3 — LS-DYNA `.k` Reader & Voxelizer
Parses `*NODE` and `*ELEMENT_*` keywords from the `.k` file.  
Produces:
- `nodes_original` — raw node coordinates as numpy array (N×3), used later for warping
- `node_id_map` — dict mapping LS-DYNA node IDs to row indices in `nodes_original`
- `moving_raw` — binary SimpleITK volume (voxelized mesh bounding box) used for registration

In [ ]:
import numpy as np
import SimpleITK as sitk

# ─────────────────────────────────────────────────────────────────────────────
# 3a  Parse LS-DYNA .k file
# ─────────────────────────────────────────────────────────────────────────────

def parse_k_file(k_path):
    """
    Parse an LS-DYNA keyword (.k) file.

    Returns
    -------
    nodes : np.ndarray, shape (N, 3)  — XYZ in mm (float64)
    node_ids : np.ndarray, shape (N,) — original LS-DYNA node IDs (int)
    node_id_map : dict                — {lsdyna_id: row_index}
    raw_lines : list[str]             — every line in the file (for round-trip writing)
    node_line_indices : dict          — {lsdyna_id: line_index_in_raw_lines}
    """
    with open(k_path, 'r', errors='replace') as fh:
        raw_lines = fh.readlines()

    nodes = []
    node_ids = []
    node_id_map = {}
    node_line_indices = {}

    in_node_block = False

    for line_idx, line in enumerate(raw_lines):
        stripped = line.strip()

        # Detect keyword cards
        if stripped.startswith('*'):
            in_node_block = stripped.upper().startswith('*NODE')
            continue

        # Skip comment lines
        if stripped.startswith('$') or stripped == '':
            continue

        if in_node_block:
            # LS-DYNA *NODE fixed-width format:
            # NID(8) X(16) Y(16) Z(16) [TC(8) RC(8)]  — columns are fixed width
            # Also handle free-format (comma-separated) gracefully.
            try:
                if ',' in stripped:
                    # Free format
                    parts = stripped.split(',')
                    nid = int(parts[0].strip())
                    x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                else:
                    # Fixed format: NID is cols 0-7, X 8-23, Y 24-39, Z 40-55
                    nid = int(line[0:8])
                    x   = float(line[8:24])
                    y   = float(line[24:40])
                    z   = float(line[40:56])

                row = len(nodes)
                nodes.append([x, y, z])
                node_ids.append(nid)
                node_id_map[nid] = row
                node_line_indices[nid] = line_idx

            except (ValueError, IndexError):
                # Non-node data line inside block — skip
                continue

    nodes_arr   = np.array(nodes,    dtype=np.float64)
    node_ids_arr = np.array(node_ids, dtype=np.int64)

    print(f"✅ Parsed {len(nodes_arr):,} nodes from {k_path}")
    print(f"   Bounding box X: [{nodes_arr[:,0].min():.2f}, {nodes_arr[:,0].max():.2f}]")
    print(f"   Bounding box Y: [{nodes_arr[:,1].min():.2f}, {nodes_arr[:,1].max():.2f}]")
    print(f"   Bounding box Z: [{nodes_arr[:,2].min():.2f}, {nodes_arr[:,2].max():.2f}]")

    return nodes_arr, node_ids_arr, node_id_map, raw_lines, node_line_indices


# ─────────────────────────────────────────────────────────────────────────────
# 3b  Voxelize node cloud → SimpleITK binary volume
# ─────────────────────────────────────────────────────────────────────────────

def voxelize_nodes(nodes, voxel_size=1.0, margin_mm=5.0):
    """
    Convert a point cloud of node coordinates into a binary voxel volume.

    Each node contributes a single lit voxel. A small margin is added around
    the bounding box so the mesh is not clipped at the volume edge.

    Parameters
    ----------
    nodes      : (N,3) float array — XYZ coordinates in mm
    voxel_size : float             — isotropic voxel spacing in mm
    margin_mm  : float             — padding around bounding box in mm

    Returns
    -------
    sitk.Image  — binary Float32 volume (1 = node present, 0 = empty)
    origin      — (3,) array — world-space origin of the volume (mm)
    """
    mins = nodes.min(axis=0) - margin_mm
    maxs = nodes.max(axis=0) + margin_mm

    # Volume dimensions in voxels
    dims = np.ceil((maxs - mins) / voxel_size).astype(int)
    volume = np.zeros(dims[::-1], dtype=np.float32)  # SimpleITK uses (z,y,x) shape

    # Map node XYZ → voxel indices
    indices = np.floor((nodes - mins) / voxel_size).astype(int)
    # Clamp to valid range
    indices = np.clip(indices, 0, dims - 1)

    # Light up voxels
    volume[indices[:,2], indices[:,1], indices[:,0]] = 1.0

    img = sitk.GetImageFromArray(volume)
    img.SetSpacing([voxel_size, voxel_size, voxel_size])
    img.SetOrigin(mins.tolist())

    print(f"✅ Voxelized mesh → volume size {img.GetSize()}, spacing {voxel_size} mm")
    return img, mins


# ── Run ───────────────────────────────────────────────────────────────────────
nodes_original, node_ids, node_id_map, k_raw_lines, node_line_indices = parse_k_file(MRI_PATH)
moving_raw, mesh_origin = voxelize_nodes(nodes_original, voxel_size=VOXEL_SIZE_MM)

# Save voxelized mesh as NIfTI for debugging / 3D Slicer review
sitk.WriteImage(moving_raw, os.path.join(OUTPUT_DIR, 'mesh_voxelized.nii.gz'))
print(f"   Saved voxelized mesh → {OUTPUT_DIR}/mesh_voxelized.nii.gz")
print(f"\n📌 nodes_original shape : {nodes_original.shape}")

## 🔧 Cell 4 — Imports & Helper Functions
All shared functions live here. Run once, use everywhere below.

In [ ]:
import os
import numpy as np
import SimpleITK as sitk
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def fix_spacing(img, default_spacing=1.0):
    spacing = list(img.GetSpacing())
    corrected = [s if s > 0.001 else default_spacing for s in spacing]
    if corrected != spacing:
        print(f"⚠️  Corrected spacing: {spacing} → {corrected}")
        img.SetSpacing(corrected)
    return img


def resample_isotropic(image, voxel_size=1.0, interpolator=sitk.sitkLinear):
    orig_spacing = image.GetSpacing()
    orig_size    = image.GetSize()
    new_size = [
        int(round(orig_size[i] * (orig_spacing[i] / voxel_size)))
        for i in range(3)
    ]
    return sitk.Resample(
        image, new_size, sitk.Transform(),
        interpolator, image.GetOrigin(),
        [voxel_size] * 3, image.GetDirection(),
        0.0, image.GetPixelID()
    )


def normalize_intensity(image):
    arr  = sitk.GetArrayFromImage(image).astype(np.float32)
    vmin, vmax = arr.min(), arr.max()
    if vmax - vmin < 1e-6:
        return image
    out = sitk.GetImageFromArray((arr - vmin) / (vmax - vmin))
    out.CopyInformation(image)
    return out


def print_image_info(name, img):
    print(f"{name}:")
    print(f"  Size    : {img.GetSize()}")
    print(f"  Spacing : {[round(s,3) for s in img.GetSpacing()]}")
    print(f"  Origin  : {[round(o,2) for o in img.GetOrigin()]}")
    arr = sitk.GetArrayFromImage(img)
    nz  = int(np.count_nonzero(arr))
    tot = int(arr.size)
    print(f"  Min/Max : {arr.min():.4f} / {arr.max():.4f}")
    print(f"  Nonzero : {nz:,} / {tot:,} ({100*nz/tot:.1f}%)")


def show_overlay(fixed, moving_reg, title="Registration Result", slices=None):
    def norm(a):
        a = a.astype(np.float32)
        mn, mx = a.min(), a.max()
        return (a - mn) / (mx - mn + 1e-8)

    f_arr = norm(sitk.GetArrayFromImage(fixed))
    m_arr = norm(sitk.GetArrayFromImage(moving_reg))

    z_mid = f_arr.shape[0] // 2
    y_mid = f_arr.shape[1] // 2
    x_mid = f_arr.shape[2] // 2
    if slices is None:
        slices = [z_mid, y_mid, x_mid]

    views = [
        (f_arr[slices[0], :, :], m_arr[slices[0], :, :], "Axial"),
        (f_arr[:, slices[1], :], m_arr[:, slices[1], :], "Coronal"),
        (f_arr[:, :, slices[2]], m_arr[:, :, slices[2]], "Sagittal"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, (ct_sl, mri_sl, label) in zip(axes, views):
        rgb = np.zeros((*ct_sl.shape, 3), dtype=np.float32)
        rgb[..., 0] = ct_sl
        rgb[..., 1] = mri_sl
        ax.imshow(rgb, origin='lower')
        ax.set_title(f"{label}  |  🔴 CT  🟢 Mesh  🟡 Overlap")
        ax.axis('off')
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f"{title.replace(' ', '_')}.png")
    plt.savefig(out, dpi=120)
    plt.show()
    print(f"✅ Saved overlay to: {out}")


metric_log = []
def log_metric(method):
    v = method.GetMetricValue()
    i = method.GetOptimizerIteration()
    metric_log.append((i, v))
    if i % 20 == 0:
        print(f"   Iter {i:04d}  |  Metric: {v:.6f}")


def plot_convergence(title="Convergence", color='steelblue'):
    if not metric_log:
        print("No metric log to plot.")
        return
    iters, vals = zip(*metric_log)
    plt.figure(figsize=(8, 3))
    plt.plot(iters, vals, color=color)
    plt.xlabel('Iteration')
    plt.ylabel('MI Metric')
    plt.title(f'{title} (more negative = better)')
    plt.tight_layout()
    plt.show()


def checkerboard(a, b, block=20):
    out = np.zeros_like(a)
    for i in range(0, a.shape[0], block):
        for j in range(0, a.shape[1], block):
            if ((i // block) + (j // block)) % 2 == 0:
                out[i:i+block, j:j+block] = a[i:i+block, j:j+block]
            else:
                out[i:i+block, j:j+block] = b[i:i+block, j:j+block]
    return out


print("✅ All helper functions loaded.")

## 🔬 Cell 5 — Preprocess: Load CT → Resample → Normalize → Center Align
The `moving` image here is the voxelized `.k` mesh from Cell 3.

In [ ]:
print("📂 Loading CT (fixed)...")
fixed_raw = sitk.ReadImage(CT_PATH, sitk.sitkFloat32)
print_image_info("CT  (raw)", fixed_raw)
print()
print_image_info(".k mesh voxelized (raw)", moving_raw)

# Fix any bad spacing values
fixed_raw  = fix_spacing(fixed_raw)
moving_raw_fixed = fix_spacing(moving_raw)

# Resample to 1 mm isotropic
print("\n🔄 Stage 1a: Resampling to 1.0 mm isotropic...")
fixed  = resample_isotropic(fixed_raw,        voxel_size=1.0)
moving = resample_isotropic(moving_raw_fixed,  voxel_size=1.0)
print(f"   CT    → {fixed.GetSize()}")
print(f"   Mesh  → {moving.GetSize()}")

# Normalize intensity to [0, 1]
print("\n🔄 Stage 1b: Normalizing intensity to [0, 1]...")
fixed  = normalize_intensity(fixed)
moving = normalize_intensity(moving)
print("   Done.")

# Save preprocessed volumes
sitk.WriteImage(fixed,  os.path.join(OUTPUT_DIR, 'ct_preprocessed.nii.gz'))
sitk.WriteImage(moving, os.path.join(OUTPUT_DIR, 'mesh_preprocessed.nii.gz'))

# Stage 2: Geometric center alignment
print("\n🎯 Stage 2: Geometric Center Alignment...")
initial_tx = sitk.CenteredTransformInitializer(
    fixed, moving,
    sitk.Euler3DTransform(),
    sitk.CenteredTransformInitializerFilter.GEOMETRY
)

resampler = sitk.ResampleImageFilter()
resampler.SetReferenceImage(fixed)
resampler.SetInterpolator(sitk.sitkLinear)
resampler.SetDefaultPixelValue(0)
resampler.SetTransform(initial_tx)
moving_init = resampler.Execute(moving)
sitk.WriteImage(moving_init, os.path.join(OUTPUT_DIR, 'mesh_center_aligned.nii.gz'))

show_overlay(fixed, moving_init, title="After_Center_Alignment")
print("✅ Stage 2 (Center Alignment) complete.")

## 🔩 Cell 6 — Stage 3: Rigid Registration (6-DOF)

In [ ]:
print("🔩 Stage 3: Rigid Registration...")
metric_log.clear()

reg_rigid = sitk.ImageRegistrationMethod()
reg_rigid.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
reg_rigid.SetMetricSamplingStrategy(reg_rigid.RANDOM)
reg_rigid.SetMetricSamplingPercentage(0.10)
reg_rigid.SetInterpolator(sitk.sitkLinear)
reg_rigid.SetShrinkFactorsPerLevel([4, 2, 1])
reg_rigid.SetSmoothingSigmasPerLevel([2, 1, 0])
reg_rigid.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
reg_rigid.SetOptimizerAsGradientDescent(
    learningRate            = 1.0,
    numberOfIterations      = 200,
    convergenceMinimumValue = 1e-6,
    convergenceWindowSize   = 10
)
reg_rigid.SetOptimizerScalesFromPhysicalShift()
reg_rigid.SetInitialTransform(initial_tx, inPlace=False)
reg_rigid.AddCommand(sitk.sitkIterationEvent, lambda: log_metric(reg_rigid))

rigid_tx = reg_rigid.Execute(fixed, moving)
print(f"\n   Metric : {reg_rigid.GetMetricValue():.6f}")
print(f"   Stop   : {reg_rigid.GetOptimizerStopConditionDescription()}")

resampler.SetTransform(rigid_tx)
moving_rigid = resampler.Execute(moving)
sitk.WriteImage(moving_rigid, os.path.join(OUTPUT_DIR, 'mesh_rigid.nii.gz'))

show_overlay(fixed, moving_rigid, title="After_Rigid_Registration")
plot_convergence("Rigid Registration Convergence", color='steelblue')
print("✅ Stage 3 (Rigid) complete.")

## 🔧 Cell 7 — Stage 4: Affine Refinement (12-DOF)

In [ ]:
print("🔧 Stage 4: Affine Refinement...")
metric_log.clear()

reg_affine = sitk.ImageRegistrationMethod()
reg_affine.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
reg_affine.SetMetricSamplingStrategy(reg_affine.RANDOM)
reg_affine.SetMetricSamplingPercentage(0.15)
reg_affine.SetInterpolator(sitk.sitkLinear)
reg_affine.SetShrinkFactorsPerLevel([2, 1])
reg_affine.SetSmoothingSigmasPerLevel([1, 0])
reg_affine.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
reg_affine.SetOptimizerAsGradientDescent(
    learningRate            = 0.5,
    numberOfIterations      = 150,
    convergenceMinimumValue = 1e-6,
    convergenceWindowSize   = 10
)
reg_affine.SetOptimizerScalesFromPhysicalShift()
reg_affine.SetInitialTransform(rigid_tx, inPlace=False)
reg_affine.AddCommand(sitk.sitkIterationEvent, lambda: log_metric(reg_affine))

affine_tx = reg_affine.Execute(fixed, moving)
print(f"\n   Metric : {reg_affine.GetMetricValue():.6f}")
print(f"   Stop   : {reg_affine.GetOptimizerStopConditionDescription()}")

resampler.SetTransform(affine_tx)
moving_affine = resampler.Execute(moving)
sitk.WriteImage(moving_affine, os.path.join(OUTPUT_DIR, 'mesh_affine.nii.gz'))
sitk.WriteImage(fixed,         os.path.join(OUTPUT_DIR, 'ct_resampled.nii.gz'))

show_overlay(fixed, moving_affine, title="After_Affine_Refinement")
plot_convergence("Affine Refinement Convergence", color='darkorange')
print("✅ Stage 4 (Affine) complete.")

## 🌀 Cell 8 — Stage 4.5: Deformable Registration (Demons)
Non-linear warp. Set `RUN_DEFORMABLE = False` to use affine result as final output.

In [ ]:
RUN_DEFORMABLE = True   # ← set False to use affine result as final output

if RUN_DEFORMABLE:
    print("🌀 Stage 4.5: Deformable Registration (SimpleITK Demons)...")

    # Ensure identical grid before Demons
    moving_on_fixed = sitk.Resample(
        moving_affine, fixed,
        sitk.Transform(), sitk.sitkLinear, 0.0, moving_affine.GetPixelID()
    )

    fixed_f32  = sitk.Cast(fixed,          sitk.sitkFloat32)
    moving_f32 = sitk.Cast(moving_on_fixed, sitk.sitkFloat32)

    print(f"   Fixed  size : {fixed_f32.GetSize()}")
    print(f"   Moving size : {moving_f32.GetSize()}")
    assert fixed_f32.GetSize() == moving_f32.GetSize(), \
        f"Size mismatch: {fixed_f32.GetSize()} vs {moving_f32.GetSize()}"

    demons = sitk.DemonsRegistrationFilter()
    demons.SetNumberOfIterations(50)
    demons.SetStandardDeviations(1.5)
    disp_field = demons.Execute(fixed_f32, moving_f32)

    field_size = disp_field.GetSize()
    print(f"   Displacement field size: {field_size}")
    assert all(s > 0 for s in field_size), \
        f"Empty displacement field {field_size} — Demons failed silently"

    disp_tx       = sitk.DisplacementFieldTransform(disp_field)
    moving_deform = sitk.Resample(
        moving, fixed, disp_tx,
        sitk.sitkLinear, 0.0, moving.GetPixelID()
    )

    sitk.WriteImage(moving_deform, os.path.join(OUTPUT_DIR, 'mesh_deformable_final.nii.gz'))
    sitk.WriteImage(
        sitk.Cast(disp_field, sitk.sitkVectorFloat32),
        os.path.join(OUTPUT_DIR, 'displacement_field.nii.gz')
    )

    show_overlay(fixed, moving_deform, title="After_Deformable_FINAL")
    FINAL_MOVING = moving_deform
    FINAL_TX     = disp_tx          # used in Cell 9 for node warping
    FINAL_TX_TYPE = 'deformable'
    print("✅ Stage 4.5 (Deformable) complete.")

else:
    FINAL_MOVING  = moving_affine
    FINAL_TX      = affine_tx
    FINAL_TX_TYPE = 'affine'
    print("⏭️  Deformable skipped — using affine result as final output.")

print(f"\n📌 FINAL_TX_TYPE = {FINAL_TX_TYPE}")

## 🔩 Cell 9 — Apply Transform to `.k` Mesh Nodes → Write New `.k` File
This is the key step unique to mesh-based registration.

### What happens here
1. Every node XYZ from `nodes_original` is passed through `FINAL_TX.TransformPoint()`
2. The transformed coordinates replace the original XYZ in the raw `.k` file lines
3. The resulting file is written as `<stem>_registered.k`

> **Note on composite transforms**: If you ran both affine + deformable stages,
> `FINAL_TX` is the Demons displacement field transform. To apply the *full* pipeline
> (initial alignment + rigid + affine + deformable) to the nodes you must compose all
> transforms in order. The cell below does this automatically.

In [ ]:
import copy

# ─────────────────────────────────────────────────────────────────────────────
# 9a  Build composite transform  (initial → rigid → affine → [deformable])
# ─────────────────────────────────────────────────────────────────────────────

def build_composite_transform(initial_tx, rigid_tx, affine_tx,
                               deformable_tx=None):
    """
    Compose all registration stages into a single transform object.

    SimpleITK CompositeTransform applies transforms from *last added* to
    *first added* (i.e., right-to-left composition), so we add in reverse
    pipeline order: deformable first, initial last.

    Returns a sitk.CompositeTransform.
    """
    composite = sitk.CompositeTransform(3)  # 3-D

    # Add in pipeline order (SimpleITK will apply last-added first)
    composite.AddTransform(initial_tx)
    composite.AddTransform(rigid_tx)
    composite.AddTransform(affine_tx)
    if deformable_tx is not None:
        composite.AddTransform(deformable_tx)

    return composite


if FINAL_TX_TYPE == 'deformable':
    composite_tx = build_composite_transform(initial_tx, rigid_tx, affine_tx, disp_tx)
else:
    composite_tx = build_composite_transform(initial_tx, rigid_tx, affine_tx)

print(f"✅ Composite transform built ({FINAL_TX_TYPE} as final stage)")


# ─────────────────────────────────────────────────────────────────────────────
# 9b  Warp all node coordinates
# ─────────────────────────────────────────────────────────────────────────────

print(f"🔄 Warping {len(nodes_original):,} nodes...")

nodes_registered = np.zeros_like(nodes_original)
for i, (x, y, z) in enumerate(nodes_original):
    nodes_registered[i] = composite_tx.TransformPoint((float(x), float(y), float(z)))

# Displacement statistics
displacements = np.linalg.norm(nodes_registered - nodes_original, axis=1)
print(f"   Mean displacement : {displacements.mean():.3f} mm")
print(f"   Max  displacement : {displacements.max():.3f} mm")
print(f"   Std  displacement : {displacements.std():.3f} mm")


# ─────────────────────────────────────────────────────────────────────────────
# 9c  Write new .k file with updated node coordinates
# ─────────────────────────────────────────────────────────────────────────────

def write_registered_k(raw_lines, node_line_indices, node_ids,
                        nodes_registered, output_path):
    """
    Write a new LS-DYNA .k file identical to the input except that
    *NODE coordinates are replaced with the registered positions.

    The fixed-width LS-DYNA format is preserved:
      NID  : cols  0-7   (I8)
      X    : cols  8-23  (E16.9)
      Y    : cols 24-39  (E16.9)
      Z    : cols 40-55  (E16.9)
      rest : cols 56+    (TC / RC constraints — preserved verbatim)
    """
    new_lines = list(raw_lines)   # shallow copy

    for row_idx, nid in enumerate(node_ids):
        line_idx = node_line_indices[nid]
        original_line = raw_lines[line_idx]

        x, y, z = nodes_registered[row_idx]

        # Preserve any trailing constraint columns (TC, RC) verbatim
        suffix = original_line[56:].rstrip('\n') if len(original_line) > 56 else ''

        new_line = (
            f"{nid:>8d}"
            f"{x:>16.9E}"
            f"{y:>16.9E}"
            f"{z:>16.9E}"
            f"{suffix}\n"
        )
        new_lines[line_idx] = new_line

    with open(output_path, 'w') as fh:
        fh.writelines(new_lines)

    print(f"✅ Registered .k file written → {output_path}")


# Determine output path
stem = os.path.splitext(os.path.basename(MRI_PATH))[0]  # e.g. 'Head_V6'
out_k_path = os.path.join(OUTPUT_DIR, f"{stem}_registered.k")

write_registered_k(
    k_raw_lines,
    node_line_indices,
    node_ids,
    nodes_registered,
    out_k_path
)

# Quick sanity check — re-parse the output and compare node count
nodes_check, _, _, _, _ = parse_k_file(out_k_path)
print(f"\n🔍 Sanity check: original {len(nodes_original):,} nodes  →  output {len(nodes_check):,} nodes")
assert len(nodes_check) == len(nodes_original), "Node count mismatch — check writer!"
print("✅ Node count matches. Output file is valid.")

## ✅ Cell 10 — Stage 5: Validation
Alpha blend + checkerboard views. Dice score for binary masks.

In [ ]:
# Mutual Information score
print("\n📊 Mutual Information Score:")
try:
    f_arr = sitk.GetArrayFromImage(fixed).astype(np.float32)
    m_arr = sitk.GetArrayFromImage(FINAL_MOVING).astype(np.float32)

    def norm_mi(a):
        vmin, vmax = a.min(), a.max()
        return (a - vmin) / (vmax - vmin + 1e-8)

    f_norm = norm_mi(f_arr).ravel()
    m_norm = norm_mi(m_arr).ravel()

    n_bins = 64
    joint_hist, _, _ = np.histogram2d(f_norm, m_norm, bins=n_bins, range=[[0,1],[0,1]])
    joint_hist = joint_hist / joint_hist.sum()

    p_f = joint_hist.sum(axis=1)
    p_m = joint_hist.sum(axis=0)

    outer   = np.outer(p_f, p_m)
    nonzero = joint_hist > 0
    mi_score = float(np.sum(
        joint_hist[nonzero] * np.log(joint_hist[nonzero] / (outer[nonzero] + 1e-12))
    ))

    h_f = -np.sum(p_f[p_f>0] * np.log(p_f[p_f>0]))
    h_m = -np.sum(p_m[p_m>0] * np.log(p_m[p_m>0]))
    nmi  = mi_score / (np.sqrt(h_f * h_m) + 1e-12)

    print(f"   MI  (raw)        : {mi_score:.4f} nats")
    print(f"   NMI (normalized) : {nmi:.4f}  [0=no overlap, 1=perfect]")
    if nmi >= 0.7:   print("   ✅ PASS — strong MI (>= 0.7)")
    elif nmi >= 0.4: print("   ⚠️  MARGINAL — partial overlap (0.4–0.7)")
    else:            print("   ❌ FAIL — weak MI (< 0.4). Check alignment.")

except Exception as e:
    print(f"   ⚠️  Could not compute MI: {e}")

In [ ]:
print("🔍 Stage 5: Validation...")

def norm_display(a):
    a = a.astype(np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

ct_arr  = norm_display(sitk.GetArrayFromImage(fixed))
mri_arr = norm_display(sitk.GetArrayFromImage(FINAL_MOVING))

z = ct_arr.shape[0] // 2
y = ct_arr.shape[1] // 2
x = ct_arr.shape[2] // 2

ct_slices  = [ct_arr[z, :, :],  ct_arr[:, y, :],  ct_arr[:, :, x]]
mri_slices = [mri_arr[z, :, :], mri_arr[:, y, :], mri_arr[:, :, x]]
plane_labels = ["Axial", "Coronal", "Sagittal"]

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Stage 5 Validation — CT (🔴) vs Registered Mesh (🟢)",
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig)

for col, (ct_sl, mri_sl, label) in enumerate(zip(ct_slices, mri_slices, plane_labels)):
    ax = fig.add_subplot(gs[0, col])
    rgb = np.zeros((*ct_sl.shape, 3), dtype=np.float32)
    rgb[..., 0] = ct_sl
    rgb[..., 1] = mri_sl
    ax.imshow(rgb, origin='lower')
    ax.set_title(f"Alpha Blend — {label}")
    ax.axis('off')

for col, (ct_sl, mri_sl, label) in enumerate(zip(ct_slices, mri_slices, plane_labels)):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(checkerboard(ct_sl, mri_sl, block=20), cmap='gray', origin='lower')
    ax.set_title(f"Checkerboard — {label}")
    ax.axis('off')

plt.tight_layout()
val_path = os.path.join(OUTPUT_DIR, 'validation_result.png')
plt.savefig(val_path, dpi=130)
plt.show()
print(f"✅ Validation image saved: {val_path}")

print("\n📊 Computing Dice Score...")
try:
    ct_bin  = sitk.Cast(sitk.BinaryThreshold(fixed,        0.1, 1.0, 1, 0), sitk.sitkUInt8)
    mri_bin = sitk.Cast(sitk.BinaryThreshold(FINAL_MOVING, 0.1, 1.0, 1, 0), sitk.sitkUInt8)
    overlap_filter = sitk.LabelOverlapMeasuresImageFilter()
    overlap_filter.Execute(ct_bin, mri_bin)
    dice = overlap_filter.GetDiceCoefficient()
    print(f"\n   🎯 Dice Score: {dice:.4f}")
    if dice >= 0.85:   print("   ✅ PASS — Excellent overlap (≥ 0.85)")
    elif dice >= 0.70: print("   ⚠️  MARGINAL — Acceptable but consider re-running")
    else:              print("   ❌ FAIL — Poor overlap. Check input images and re-run.")
except Exception as e:
    print(f"   ⚠️  Could not compute Dice: {e}")

print("\n" + "=" * 60)
print("📦 OUTPUT FILES:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    fsize = os.path.getsize(fpath) / 1e6
    print(f"   {fname:<50} {fsize:.1f} MB")
print("=" * 60)
print("\n🎯 KEY OUTPUT:")
print(f"   {stem}_registered.k  ← registered LS-DYNA mesh (updated node XYZ)")
print("   mesh_deformable_final.nii.gz / mesh_affine.nii.gz  ← voxel view")
print("   displacement_field.nii.gz  ← Demons warp field")
print("\n   Load ct_resampled.nii.gz + mesh_deformable_final.nii.gz")
print("   into 3D Slicer to verify alignment visually.")